- Generate a data suitable for de-duplication task using AI
- Different types of duplication
- different approached to detect duplications
- create a systematic approach to handle duplicated data in a dataset

# **De-Duplication:**

**Different Types of Duplicates:**

- Exact Duplicates
- Partial Duplicates
- Near duplicates (Fuzzy Duplicates)
- Cross-Column Duplicates
- Sentiment Duplicates
- Time-Based Duplicates

In [88]:
import pandas as pd

df = pd.read_csv("/mnt/d/Pytopia/Machine Learning/Personal_ML_Notes/EDA/data/customer_dedup_dataset.csv")
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
1,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
2,1002,Emily Watson,Emily,Watson,28,Manchester,emily.watson@email.com,7700900002
3,1002,Emilly Watson,Emilly,Watson,28,Manchester,emily.watson@email.com,7700900002
4,1003,Mohammed Al-Rashid,Mohammed,Al-Rashid,45,Birmingham,m.alrashid@email.com,7700900003
5,1003,Mohammad Al Rashid,Mohammad,Al Rashid,45,Birmingham,m.alrashid@email.com,7700900003
6,1004,Sarah O'Brien,Sarah,O'Brien,31,Leeds,sarah.obrien@email.com,7700900004
7,1004,Sarah OBrien,Sarah,OBrien,31,Leeds,sarah.obrien@email.com,7700900004
8,1005,Thomas Nguyen,Thomas,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005
9,1005,Tom Nguyen,Tom,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005


This dataset contains information about a small e-commerce shop and we wanna preprocess this dataset and then do a de-duplication task on it.

## 1.1. Initial Preprocessing:

In [89]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  41 non-null     int64
 1   full_name    41 non-null     str  
 2   first_name   41 non-null     str  
 3   last_name    41 non-null     str  
 4   age          41 non-null     int64
 5   city         41 non-null     str  
 6   email        41 non-null     str  
 7   phone        41 non-null     int64
dtypes: int64(3), str(5)
memory usage: 2.7 KB


In [90]:
# data types
df.dtypes

customer_id    int64
full_name        str
first_name       str
last_name        str
age            int64
city             str
email            str
phone          int64
dtype: object

In [91]:
# normalizing the strings
df['full_name'] = df['full_name'].str.lower()
df['city'] = df['city'].str.lower()
df['email'] = df['email'].str.lower()
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,Emily,Watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,Emilly,Watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,Mohammed,Al-Rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,Mohammad,Al Rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,Sarah,O'Brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,Sarah,OBrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,Thomas,Nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,Tom,Nguyen,52,liverpool,t.nguyen@email.com,7700900005


In [92]:
# extracting domain and username from the email is possible
df['email_username'] = df['email'].apply(lambda x: x.split("@")[0])
df['domain'] = df['email'].apply(lambda x: x.split("@")[1])
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone,email_username,domain
0,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001,james.carter,email.com
1,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001,james.carter,email.com
2,1002,emily watson,Emily,Watson,28,manchester,emily.watson@email.com,7700900002,emily.watson,email.com
3,1002,emilly watson,Emilly,Watson,28,manchester,emily.watson@email.com,7700900002,emily.watson,email.com
4,1003,mohammed al-rashid,Mohammed,Al-Rashid,45,birmingham,m.alrashid@email.com,7700900003,m.alrashid,email.com
5,1003,mohammad al rashid,Mohammad,Al Rashid,45,birmingham,m.alrashid@email.com,7700900003,m.alrashid,email.com
6,1004,sarah o'brien,Sarah,O'Brien,31,leeds,sarah.obrien@email.com,7700900004,sarah.obrien,email.com
7,1004,sarah obrien,Sarah,OBrien,31,leeds,sarah.obrien@email.com,7700900004,sarah.obrien,email.com
8,1005,thomas nguyen,Thomas,Nguyen,52,liverpool,t.nguyen@email.com,7700900005,t.nguyen,email.com
9,1005,tom nguyen,Tom,Nguyen,52,liverpool,t.nguyen@email.com,7700900005,t.nguyen,email.com


In [93]:
# restructuring the data to evoid cross-column duplicates
df = df[['customer_id', 'full_name', 'age', 'city', 'email', 'phone']]
df.head(10)

,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005


## 1.2. De-Duplication:

In [94]:
# number of the unique records in every variable
df.nunique()

customer_id    24
full_name      36
age            24
city           25
email          25
phone          24
dtype: int64

By seeing thet there is 24 unique customer ids we can say there are just `24 records` out of 50 without duplicated records.

In [95]:
# exact duplicates
df[df.duplicated(keep=False)]

,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,34,london,james.carter@email.com,7700900001
32,1019,nathan brooks,31,plymouth,n.brooks@email.com,7700900019
33,1019,nathan brooks,31,plymouth,n.brooks@email.com,7700900019


In [97]:
df = df.drop_duplicates()
print(f"shape of the dataset: {df.shape}")
df.head(10)

shape of the dataset: (39, 6)


,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005
10,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006


In [105]:
# potential partial duplicates
def potential_partial_duplicates(subset):
	return df[df.duplicated(subset=subset, keep=False)]

potential_partial_duplicates(["customer_id", 'phone'])

,customer_id,full_name,age,city,email,phone
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005
10,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006
11,1006,charlote evans,23,bristol,c.evans@email.com,7700900006


It's better to analyze them using fuzzy duplicate techniques: